# Piloto de fine-tuning — acelerar sin cambiar el tono

Verificación rápida: coge audios ya generados en Drive (por defecto, las frases de prueba del LoRA de Qwen) y
los acelera un poco **conservando el tono**, con Rubber Band. Los escuchas aquí mismo, uno debajo de otro, con
su velocidad en caracteres por segundo. Como referencia, Naval en inglés habla a unos 19,5 c/s y Qwen en
español sale a unos 15.

No genera nada con el modelo ni guarda nada en Drive, salvo que pongas `GUARDAR = True`.

**Entorno: CPU, sin GPU** (*Entorno de ejecución → Cambiar tipo de entorno → CPU*). Tarda un par de minutos.
Ejecuta las celdas en orden; la última desconecta el entorno.

In [ ]:
# 1. Instalación: Rubber Band, el programa que cambia el tempo sin tocar el tono (no hace falta reiniciar)
!apt-get -qq install -y rubberband-cli > /dev/null && rubberband --version 2>&1 | head -1

In [ ]:
# 2. Parámetros
CARPETA = "piloto/lora_qwen/B/prueba"     # dentro de fine_tunned_naval
VERSIONES = ["paso_0158"]                 # añade "base" para comparar con el modelo sin LoRA
FRASES = ["n12", "n18", "n25"]
TEMPOS = [1.00, 1.05, 1.10, 1.15]         # 1.10 = un 10 % más rápido
GUARDAR = False                           # True: deja los acelerados en <CARPETA>/tempo/ en Drive

In [ ]:
# 3. Preparar: código del repo, Drive y los textos de las frases
import os, sys, subprocess
from pathlib import Path
RAMA = "claude/quirky-dirac-x18c3n"
if os.path.isdir("/content/repo"):
    subprocess.run(["git", "-C", "/content/repo", "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", RAMA, "https://github.com/DreiSup/spanishnav.al-podcast", "/content/repo"], check=True)
sys.path[:0] = ["/content/repo/criba", "/content/repo/piloto"]
import comun, tempo

comun.montar_drive()
BASE = comun.DRIVE / CARPETA
items, _ = comun.cargar_items()
TEXTO = {it["id"]: " ".join(it["frases"]) for it in items}
faltan = [str(BASE / v / f"{f}.wav") for v in VERSIONES for f in FRASES if not (BASE / v / f"{f}.wav").is_file()]
if faltan:
    raise SystemExit(f"Faltan audios en Drive, por ejemplo {faltan[0]}")
print("RESULTADO: OK")

In [ ]:
# 4. Acelerar y escuchar
import soundfile as sf
from IPython.display import Audio, Markdown, display

for f in FRASES:
    display(Markdown(f"### {f}\n*{TEXTO[f]}*"))
    for v in VERSIONES:
        audio, sr = sf.read(str(BASE / v / f"{f}.wav"), dtype="float32")
        for t in TEMPOS:
            rapido, metodo = tempo.acelerar(audio, sr, t)
            cps = len(TEXTO[f]) / (len(rapido) / sr)
            display(Markdown(f"**{v} · ×{t:.2f}** — {len(rapido) / sr:.1f} s, {cps:.1f} c/s ({metodo})"))
            display(Audio(rapido, rate=sr))
            if GUARDAR and t != 1.0:
                destino = BASE / "tempo" / v
                destino.mkdir(parents=True, exist_ok=True)
                sf.write(str(destino / f"{f}_x{t:.2f}.wav"), rapido, sr, subtype="PCM_16")

In [ ]:
# 5. Desconectar (sube a Drive lo pendiente si has guardado algo)
comun.desconectar()